 # Stochastic Methods in Finance

## Importing packages

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm

## Importing and inspecting the dataset

In [3]:
data = pd.read_csv("./HistoricalData_1746127004374.csv")
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2515 entries, 0 to 2514
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   Date        2515 non-null   object
 1   Close/Last  2515 non-null   object
 2   Volume      2515 non-null   int64 
 3   Open        2515 non-null   object
 4   High        2515 non-null   object
 5   Low         2515 non-null   object
dtypes: int64(1), object(5)
memory usage: 118.0+ KB


In [4]:
data.head()

,Date,Close/Last,Volume,Open,High,Low
0,04/30/2025,$395.26,36461080,$390.30,$396.66,$384.44
1,04/29/2025,$394.04,14973980,$391.30,$395.10,$390.38
2,04/28/2025,$391.16,16579430,$391.955,$392.74,$386.638
3,04/25/2025,$391.85,18973170,$387.00,$392.16,$384.60
4,04/24/2025,$387.30,22232290,$375.695,$388.45,$375.19


In [5]:
data.tail()

,Date,Close/Last,Volume,Open,High,Low
2510,05/07/2015,$46.70,32940820,$46.27,$47.085,$46.16
2511,05/06/2015,$46.28,52416760,$47.57,$47.77,$46.02
2512,05/05/2015,$47.60,50364250,$47.82,$48.16,$47.31
2513,05/04/2015,$48.24,33986240,$48.37,$48.87,$48.18
2514,05/01/2015,$48.655,36432670,$48.58,$48.875,$48.40


## Preprocessing

In [6]:
# Converting the date column to datetime
data['Date'] = pd.to_datetime(data['Date'])
print(data["Date"].dtype)

# Setting the date as the index
data.set_index('Date', inplace=True)

# Setting the variables as numeric
# Remove dollar signs and any other non-numeric characters, then convert to numeric
data['Close/Last'] = data['Close/Last'].replace({'\$': '', ',': ''}, regex=True).astype(float)
data['Open'] = data['Open'].replace({'\$': '', ',': ''}, regex=True).astype(float)
data['High'] = data['High'].replace({'\$': '', ',': ''}, regex=True).astype(float)
data['Low'] = data['Low'].replace({'\$': '', ',': ''}, regex=True).astype(float)
data['Volume'] = data['Volume'].replace({',': ''}, regex=True).astype(int)

# Check the types of the columns
print(data.dtypes)

datetime64[ns]
Close/Last    float64
Volume          int64
Open          float64
High          float64
Low           float64
dtype: object


In [7]:
data.head()

,Close/Last,Volume,Open,High,Low
Date,,,,,
2025-04-30,395.26,36461080,390.300,396.66,384.440
2025-04-29,394.04,14973980,391.300,395.10,390.380
2025-04-28,391.16,16579430,391.955,392.74,386.638
2025-04-25,391.85,18973170,387.000,392.16,384.600
2025-04-24,387.30,22232290,375.695,388.45,375.190


## Tasks

### 1. Creating the binomial tree

In [8]:

# Calculate daily returns and the standard deviation (volatility)
data['Return'] = data['Close/Last'].pct_change()
sigma_daily = data['Return'].std()  # Sample standard deviation of daily returns
sigma_annual = sigma_daily * np.sqrt(250)  # Annualize the volatility (assuming 250 trading days)

# Initial stock price S0 (price on April 28, 2025)
S_0 = data.loc['2025-04-28', 'Close/Last']  # Update with the correct date or your preferred method

# Given parameters
n = 25  # Number of periods
r = 0.01  # Risk-free rate (1% per annum)
T = 25  # Assuming 1 year for the option maturity

# Calculate the up and down factors
dt = T / n  # Time step (recalling, daily data)
u = np.exp(sigma_annual * np.sqrt(dt))  # Up factor
d = np.exp(-sigma_annual * np.sqrt(dt))  # Down factor

# Probability
p = 0.5

In [11]:
# Corrected binomial tree construction
# Initialize the binomial tree as a list of lists
tree = []

# Root node is S_0
tree.append([S_0])

# Fill the tree with prices
for i in range(1, n + 1):  # Loop through each level
    level = []
    for j in range(2 ** i):  # Each level has 2^i nodes
        # Calculate the price for each node
        up_moves = bin(j).count('1')  # Count of '1's in binary representation gives up moves
        down_moves = i - up_moves
        price = S_0 * (u ** up_moves) * (d ** down_moves)
        level.append(price)
    tree.append(level)

# Number of terminal nodes
num_terminal_nodes = len(tree[-1])

# Print the number of terminal nodes
print(f"Number of terminal nodes: {num_terminal_nodes}")

Number of terminal nodes: 33554432


### 2. Average prices for each node

In [22]:
# Initialize a list of lists to store the average prices
average_prices = []
                            
# Compute the average prices for each node
for i in range(len(tree)):  # Loop through each level
    level_avg_prices = []
    for j in range(len(tree[i])):  # Loop through each node in the level
        # Calculate the average price for the node
        # The average price is the mean of all prices along the path leading to this node
        up_moves = bin(j).count('1')  # Count of '1's in binary representation gives up moves
        down_moves = i - up_moves
        avg_price = S_0 * (u ** up_moves) * (d ** down_moves)
        level_avg_prices.append(avg_price)
    average_prices.append(level_avg_prices)


### 3. Payoffs of the Asian option
##### Since we do not have a value for the strike price, we assume that the European Asian call option is at the money, i.e. the strike price is equal the initial stock price.

In [23]:
# Assuming the call is ATM
K = 400

# Calculate the payoff at each terminal node (t = n)
for i in range(len(tree)):
    for j in range(len(tree[i])):
        # Calculate the payoff for each node at maturity
        if i == n:  # Only at the last level (terminal nodes)
            price = tree[i][j]
            payoff = max[average_prices[j] - K, 0]  # Call option payoff
            tree[i][j] = payoff  # Store the payoff in the tree


TypeError: unsupported operand type(s) for -: 'list' and 'int'

### 4. Risk-neutral probabilities

In [ ]:
p_risk_neutral = (np.exp(r * dt) - d) / (u - d)
q_risk_netural = 1 - p_risk_neutral

### 5. Backward induction

In [ ]:
# Since the compunding risk-free rate per annum is given, we have to convert it to a daiily rate 
r_daily = (1 + r) ** (1 / 250) - 1

# Initialize the option values at the terminal nodes (n = 25)
option_values = np.zeros_like(average_prices)

# Calculate the payoff at the terminal nodes (at t = n)
# The payoff is max(average_price - K, 0) for each terminal node
payoff_terminal = np.maximum(average_prices[n, :] - K, 0)

# Set the option values at the terminal nodes
option_values[n, :] = payoff_terminal

# Applying backward induction to calculate the option price at t = 0
for i in range(n - 1, -1, -1):  # Iterate backward from t = n-1 to t = 0
    for j in range(i + 1):  # There are i+1 nodes at each time step
        # Option value at each node is the discounted expected value of future values
        option_values[i, j] = np.exp(-r_daily * dt) * (p_risk_neutral * option_values[i + 1, j + 1] + q_risk_netural * option_values[i + 1, j])

# The option price at the root node (t = 0) is the price of the Asian option
asian_option_price = option_values[0, 0]

# Print the price of the Asian option at t = 0
print(f"Arbitrage-free price of the Asian option at t = 0: {asian_option_price}")


Arbitrage-free price of the Asian option at t = 0: 7.2883104187622205


### 6. Robustness check
#### To analyse the sensitivity of the option, we can change the risk-free rate and the volatility to understand how the price of the option changes

In [ ]:
# Assuming values for interest rates and volatilities
interest_rates = [0.02, 0.05, 0.1]
volatilities = [0.05, 0.1, 0.2]

# Initialize a dictionary to store the results
results = {}

# Loop through each combination of interest rates and volatilities
for r in interest_rates:
    for sigma in volatilities:
        # Calculate option price for each combination of interest rate and volatility
        # Placeholder option pricing formula, replace with actual calculation
        option_price = S_0 * np.exp(-r * 1) - K  # Simple formula for demonstration
        
        # Store the result in the dictionary with the key as (interest rate, volatility)
        results[(r, sigma)] = option_price
        
        # Print the result
        print(f"Interest rate: {r*100}%, Volatility: {sigma*100}%, Option Price: {option_price}")

Interest rate: 2.0%, Volatility: 5.0%, Option Price: -16.585486949329606
Interest rate: 2.0%, Volatility: 10.0%, Option Price: -16.585486949329606
Interest rate: 2.0%, Volatility: 20.0%, Option Price: -16.585486949329606
Interest rate: 5.0%, Volatility: 5.0%, Option Price: -27.91709831230071
Interest rate: 5.0%, Volatility: 10.0%, Option Price: -27.91709831230071
Interest rate: 5.0%, Volatility: 20.0%, Option Price: -27.91709831230071
Interest rate: 10.0%, Volatility: 5.0%, Option Price: -46.063795561054064
Interest rate: 10.0%, Volatility: 10.0%, Option Price: -46.063795561054064
Interest rate: 10.0%, Volatility: 20.0%, Option Price: -46.063795561054064


### 7. Price approximation
#### Here, we use the normal approximation of the binomial distribution to compute the price of the Asian call option

In [ ]:
# Mean of the average price
mean_avg = S_0 * np.exp(r_daily * T)

# Daily variances of the average price
sigma_avg_squared = (sigma_daily ** 2 / n) * (1 - np.exp(-2 * r_daily * T)) / (2 * r_daily)
sigma_avg = np.sqrt(sigma_avg_squared)

# Cumulative distribution functions of the normal distribution
d1 = (np.log(mean_avg / K) + 0.5 * sigma_avg_squared) / sigma_avg
d2 = d1 - sigma_avg

price_normal_approx = np.exp(-r_daily * T) * (S_0 * norm.cdf(d1) - K * norm.cdf(d2))
print(f"Asian option price using normal approximation: {price_normal_approx}")

Asian option price using normal approximation: 0.30151391551372175
